In [1]:
import pandas as pd
import math
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv("housing.csv", sep=";")
df = df.dropna()
df = pd.get_dummies(df, columns=["ocean_proximity"])

X = df.drop("median_house_value", axis=1).values.tolist()
y = df["median_house_value"].values.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modell = LinearRegression()
modell.fit(X_train, y_train)
vorhersagen = modell.predict(X_test)

Teilaufgabe 4.1

In [2]:
korrelationen = df.corr(numeric_only=True)["median_house_value"].drop("median_house_value")
sortiert = korrelationen.abs().sort_values(ascending=False)

print("Korrelation mit median_house_value (absteigend nach Stärke):")
print()
for name in sortiert.index:
    print(name + ":", round(korrelationen[name], 4))

Korrelation mit median_house_value (absteigend nach Stärke):

median_income: 0.6884
ocean_proximity_INLAND: -0.4848
ocean_proximity_<1H OCEAN: 0.2576
ocean_proximity_NEAR BAY: 0.1605
latitude: -0.1446
ocean_proximity_NEAR OCEAN: 0.1404
total_rooms: 0.1333
housing_median_age: 0.1064
households: 0.0649
total_bedrooms: 0.0497
longitude: -0.0454
population: -0.0253
ocean_proximity_ISLAND: 0.0235


Das Feld median_income hat mit Abstand die stärkste Korrelation mit dem Hauswert (0.69). Das ergibt Sinn, weil in Gebieten wo die Leute mehr verdienen, sind die Häuser auch teurer. Auch die Meeresnähe spielt eine grosse Rolle – ocean_proximity_INLAND hat eine Korrelation von -0.48, was bedeutet dass Häuser im Inland deutlich günstiger sind als Häuser in Meeresnähe. Die geografische Lage (latitude) hat ebenfalls einen Einfluss (-0.14), da Häuser im Süden Kaliforniens tendenziell teurer sind. Felder wie population oder longitude haben kaum einen direkten Einfluss auf den Hauswert.

Teilaufgabe 4.2

In [3]:
y_mean_test = sum(y_test) / len(y_test)

ss_tot = sum((wert - y_mean_test) ** 2 for wert in y_test)
ss_res = sum((y_test[i] - vorhersagen[i]) ** 2 for i in range(len(y_test)))
r2 = 1 - (ss_res / ss_tot)

mae = sum(abs(y_test[i] - vorhersagen[i]) for i in range(len(y_test))) / len(y_test)

print("R2:", round(r2, 4))
print("MAE:", round(mae, 2))

R2: 0.6488
MAE: 50413.43


Als Messmetrik habe ich R² gewählt. Es zeigt, wie viel der Varianz des Zielfelds das Modell erklären kann. Ein Wert von 1 wäre perfekt, 0 bedeutet das Modell ist nicht besser als der Mittelwert. Zusätzlich habe ich den MAE (Mean Absolute Error) berechnet, der die durchschnittliche Abweichung in Dollar angibt.

Teilaufgabe 4.3

In [4]:
alle_y = sorted(y)
n_y = len(alle_y)
if n_y % 2 == 1:
    schwellenwert = alle_y[n_y // 2]
else:
    schwellenwert = (alle_y[n_y // 2 - 1] + alle_y[n_y // 2]) / 2

print("Schwellenwert (Median aller Hauswerte):", schwellenwert)
print()

tp = 0
tn = 0
fp = 0
fn = 0

for i in range(len(y_test)):
    tatsaechlich = 1 if y_test[i] >= schwellenwert else 0
    vorhergesagt = 1 if vorhersagen[i] >= schwellenwert else 0

    if tatsaechlich == 1 and vorhergesagt == 1:
        tp += 1
    elif tatsaechlich == 0 and vorhergesagt == 0:
        tn += 1
    elif tatsaechlich == 0 and vorhergesagt == 1:
        fp += 1
    else:
        fn += 1

print("Wahrheitsmatrix (teuer = >= Schwellenwert):")
print()
print("                    | Vorhergesagt teuer | Vorhergesagt günstig")
print("Tatsächlich teuer   |", tp, "              |", fn)
print("Tatsächlich günstig |", fp, "               |", tn)
print()

sensitivitaet = tp / (tp + fn)
spezifizitaet = tn / (tn + fp)

print("Sensitivität:", round(sensitivitaet, 4))
print("Spezifizität:", round(spezifizitaet, 4))

Schwellenwert (Median aller Hauswerte): 179700

Wahrheitsmatrix (teuer = >= Schwellenwert):

                    | Vorhergesagt teuer | Vorhergesagt günstig
Tatsächlich teuer   | 1859               | 164
Tatsächlich günstig | 612                | 1452

Sensitivität: 0.9189
Spezifizität: 0.7035


Teilaufgabe 4.4

Das Modell erreicht ein R² von ca. 0.65, es erklärt also rund 65% der Varianz im Hauswert. Der MAE liegt bei ungefähr 50'000 Dollar – das klingt viel, aber bei Hauspreisen zwischen 15'000 und 500'000 USD ist das noch in einem vertretbaren Rahmen. Das wichtigste Merkmal ist klar median_income (Korrelation 0.69), was auch Sinn macht, weil in reicheren Gegenden die Häuser einfach teurer sind. Auch die Meeresnähe hat einen grossen Einfluss, ocean_proximity_INLAND hat eine Korrelation von -0.48. Die grössten Fehler macht das Modell bei Häusern über 500'001 USD, weil dieser Wert im Datensatz gekappt ist. Vermutlich würde ein RandomForest besser abschneiden, da der Zusammenhang nicht wirklich linear ist.